# Data Fetching — Bybit (Migrated from MT5)

> **Migration note**: This notebook was migrated from `00_data_feching.ipynb` (MT5 version).
> The execution layer changed from **MetaTrader5** to **Bybit REST API** (`pybit`).
> Strategy logic is unchanged — same symbols, timeframes, and CSV output format.

Fetches OHLCV candles from Bybit Linear (USDT-perpetual) and saves to:
`./data/<SYMBOL>/<TIMEFRAME>/ohlcv.csv`

The CSV output is **column-compatible** with the MT5 version so all downstream notebooks work unchanged.

## Differences from MT5 version
| MT5 | Bybit |
|-----|-------|
| `mt5.copy_rates_from_pos()` | `pybit` kline endpoint |
| Broker symbols (`BTCUSD.i`) | Bybit symbols (`BTCUSDT`) |
| Local terminal required | REST API only |
| `tick_volume` column | `volume` column (base asset qty) |


In [1]:
# SECTION 1 — Imports and parameters
import warnings
warnings.filterwarnings("ignore")

import sys, os, time
from pathlib import Path
import pandas as pd

# Resolve project root (notebooks/ -> parent)
_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

# Load .env from project root
try:
    from dotenv import load_dotenv
    _env = _ROOT / ".env"
    if _env.exists():
        load_dotenv(_env)
        print(f"Loaded .env from {_env}")
    else:
        print(f"No .env at {_env} — set BYBIT_API_KEY / BYBIT_API_SECRET manually")
except ImportError:
    print("python-dotenv not installed; relying on environment variables")

# ── Fetch parameters ──────────────────────────────────────────────
SYMBOLS = [ "XAUUSDT","ETHUSDT","BNBUSDT","BCHUSDT","XRPUSDT","ADAUSDT","SOLUSDT","DOGEUSDT","DOTUSDT","AVAXUSDT","BCHUSDT","SHIB1000USDT","LTCUSDT","XLMUSDT"]
# Other options: "ETHUSDT", "SOLUSDT", "XRPUSDT", "BNBUSDT", "ADAUSDT" , "LNKUSDT" , "BCHUSDT" , "UNIUSDT" , "LTCUSD"
# "DOGEUSD"

TIMEFRAMES = [ "M1","M5","H1","D1"]

BYBIT_INTERVAL_MAP = {
    "M1": "1",  "M3": "3",  "M5": "5",  "M15": "15", "M30": "30",
    "H1": "60", "H2": "120","H4": "240","H6": "360", "H12": "720",
    "D1": "D",  "W1": "W",  "MN": "M",
}

BARS_BY_TF = {
    "M1": 600000, "M5": 160000, "M15":10000, "M30": 5000,
    "H1": 40000,  "H4": 1000,  "D1": 2000,
}

CATEGORY = "linear"
TESTNET  = os.getenv("BYBIT_TESTNET", "false").lower() == "true"

print(f"Symbols   : {SYMBOLS}")
print(f"Timeframes: {TIMEFRAMES}")
print(f"Testnet   : {TESTNET}")


Loaded .env from D:\bot\ema-h1trend-exchange\.env
Symbols   : ['XAUUSDT', 'ETHUSDT', 'BNBUSDT', 'BCHUSDT', 'XRPUSDT', 'ADAUSDT', 'SOLUSDT', 'DOGEUSDT', 'DOTUSDT', 'AVAXUSDT', 'BCHUSDT', 'SHIB1000USDT', 'LTCUSDT', 'XLMUSDT']
Timeframes: ['M1', 'M5', 'H1', 'D1']
Testnet   : False


In [2]:
# SECTION 2 — Bybit client setup
from pybit.unified_trading import HTTP

api_key    = os.getenv("BYBIT_API_KEY", "")
api_secret = os.getenv("BYBIT_API_SECRET", "")

client = HTTP(
    testnet=TESTNET,
    api_key=api_key or None,
    api_secret=api_secret or None,
)

print(f"Bybit client ready (testnet={TESTNET})")
print(f"API key set: {bool(api_key)}")


Bybit client ready (testnet=False)
API key set: True


In [3]:
# SECTION 3 — Fetch helpers
def fetch_bybit_klines(symbol: str, interval: str, limit: int = 1000) -> pd.DataFrame:
    """Fetch up to `limit` candles using time-based pagination (end param)."""
    all_rows = []
    end_time  = None
    remaining = limit
    MAX_RETRIES = 5

    while remaining > 0:
        batch  = min(remaining, 1000)
        kwargs = dict(category=CATEGORY, symbol=symbol, interval=interval, limit=batch)
        if end_time is not None:
            kwargs["end"] = end_time

        # retry loop for this batch
        rows = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = client.get_kline(**kwargs)
                if resp.get("retCode") != 0:
                    raise RuntimeError(f"Bybit error {resp.get('retCode')}: {resp.get('retMsg')}")
                rows = resp["result"].get("list", [])
                break  # success
            except Exception as exc:
                if attempt == MAX_RETRIES:
                    raise RuntimeError(
                        f"Failed after {MAX_RETRIES} attempts for {symbol} {interval}: {exc}"
                    ) from exc
                wait = 2 ** attempt  # 2, 4, 8, 16 seconds
                print(f"    attempt {attempt}/{MAX_RETRIES} failed ({exc}), retrying in {wait}s...")
                time.sleep(wait)

        if not rows:
            break

        all_rows.extend(rows)
        remaining -= len(rows)
        # oldest timestamp in this batch (rows are newest-first) minus 1ms to avoid overlap
        end_time = int(rows[-1][0]) - 1
        if len(rows) < batch:
            break
        time.sleep(0.05)  # rate-limit safety

    if not all_rows:
        raise RuntimeError(f"No data for {symbol} {interval}")

    df = pd.DataFrame(
        list(reversed(all_rows)),  # oldest-first
        columns=["time", "open", "high", "low", "close", "volume", "turnover"],
    )
    df["time"] = pd.to_datetime(df["time"].astype("int64"), unit="ms", utc=True)
    df = df.set_index("time").sort_index()
    df = df[~df.index.duplicated(keep="last")]  # remove boundary duplicates
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    # Drop forming (live) candle — mirrors MT5 behavior
    if len(df) > 0:
        df = df.iloc[:-1]

    return df[["open", "high", "low", "close", "volume"]]


def resolve_bybit_symbol(symbol: str) -> str | None:
    resp = client.get_instruments_info(category=CATEGORY, symbol=symbol)
    if resp.get("retCode") != 0:
        return None
    return symbol if resp.get("result", {}).get("list") else None


print("Fetch helpers defined.")


Fetch helpers defined.


In [4]:
# SECTION 4 — Fetch and cache all symbols / timeframes
BASE_DIR = Path("./data")
_TEHRAN  = "Asia/Tehran"
results  = {}

for symbol in SYMBOLS:
    resolved = resolve_bybit_symbol(symbol)
    if resolved is None:
        print(f"Skip {symbol}: not found on Bybit {CATEGORY}")
        continue

    print(f"Fetching {symbol}...")

    for tf in TIMEFRAMES:
        interval = BYBIT_INTERVAL_MAP.get(tf)
        if interval is None:
            print(f"  Skip {tf}: no interval mapping")
            continue

        bars = BARS_BY_TF.get(tf, 5000)
        try:
            df = fetch_bybit_klines(symbol, interval, limit=bars)
            # Convert index to Tehran time before persisting
            df.index = df.index.tz_convert(_TEHRAN)
            out_dir  = BASE_DIR / symbol / tf
            out_dir.mkdir(parents=True, exist_ok=True)
            out_file = out_dir / "ohlcv.csv"
            df.to_csv(out_file)
            results[(symbol, tf)] = len(df)
            print(f"  {tf}: {len(df):,} rows -> {out_file}")
        except Exception as exc:
            print(f"  {tf}: FAILED — {exc}")

print(f"\nDone. Cached {len(results)} time series.")

if results:
    sym, tf = next(iter(results))
    preview = pd.read_csv(BASE_DIR / sym / tf / "ohlcv.csv", index_col=0, parse_dates=True)
    print(f"\nPreview ({sym} {tf}):")
    display(preview.tail(5))

Fetching XAUUSDT...
  M1: 98,985 rows -> data\XAUUSDT\M1\ohlcv.csv
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
  M5: 19,798 rows -> data\XAUUSDT\M5\ohlcv.csv
  H1: 1,650 rows -> data\XAUUSDT\H1\ohlcv.csv
  D1: 68 rows -> data\XAUUSDT\D1\ohlcv.csv
Fetching ETHUSDT...
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
  M1: 599,999 rows -> data\ETHUSDT\M1\ohlcv.csv


2026-05-17 00:30:15 - pybit._http_manager - ERROR - Too many visits. Exceeded the API Rate Limit. (ErrCode: 10006). Hit the API rate limit on https://api.bybit.com/v5/market/kline?category=linear&end=1751365199999&interval=5&limit=1000&symbol=ETHUSDT. Sleeping then trying again.
2026-05-17 00:30:15 - pybit._http_manager - ERROR - API rate limit will reset at 00:30:17.009. Sleeping for 2000 ms. Retrying...
2026-05-17 00:30:17 - pybit._http_manager - ERROR - Too many visits. Exceeded the API Rate Limit. (ErrCode: 10006). Hit the API rate limit on https://api.bybit.com/v5/market/kline?category=linear&end=1751365199999&interval=5&limit=1000&symbol=ETHUSDT. Sleeping then trying again.
2026-05-17 00:30:17 - pybit._http_manager - ERROR - API rate limit will reset at 00:30:19.889. Sleeping for 2000 ms. Retrying...


  M5: 159,999 rows -> data\ETHUSDT\M5\ohlcv.csv
  H1: 39,999 rows -> data\ETHUSDT\H1\ohlcv.csv
  D1: 1,888 rows -> data\ETHUSDT\D1\ohlcv.csv
Fetching BNBUSDT...
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
  M1: 599,999 rows -> data\BNBUSDT\M1\ohlcv.csv
  M5: 159,999 rows -> data\BNBUSDT\M5\ohlcv.csv
  H1: 39,999 rows -> data\BNBUSDT\H1\ohlcv.csv
  D1: 1,782 rows -> data\BNBUSDT\D1\ohlcv.csv
Fetching BCHUSDT...
  M1: 599,999 rows -> data\BCHUSDT\M1\ohlcv.csv


2026-05-17 01:00:05 - pybit._http_manager - ERROR - Too many visits. Exceeded the API Rate Limit. (ErrCode: 10006). Hit the API rate limit on https://api.bybit.com/v5/market/kline?category=linear&end=1772966999999&interval=5&limit=1000&symbol=BCHUSDT. Sleeping then trying again.
2026-05-17 01:00:05 - pybit._http_manager - ERROR - API rate limit will reset at 01:00:07.955. Sleeping for 1999 ms. Retrying...
2026-05-17 01:00:15 - pybit._http_manager - ERROR - Too many visits. Exceeded the API Rate Limit. (ErrCode: 10006). Hit the API rate limit on https://api.bybit.com/v5/market/kline?category=linear&end=1770566999999&interval=5&limit=1000&symbol=BCHUSDT. Sleeping then trying again.
2026-05-17 01:00:15 - pybit._http_manager - ERROR - API rate limit will reset at 01:00:17.534. Sleeping for 2000 ms. Retrying...


  M5: 159,999 rows -> data\BCHUSDT\M5\ohlcv.csv
  H1: 39,999 rows -> data\BCHUSDT\H1\ohlcv.csv
  D1: 1,979 rows -> data\BCHUSDT\D1\ohlcv.csv
Fetching XRPUSDT...
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
  M1: 599,999 rows -> data\XRPUSDT\M1\ohlcv.csv
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
  M5: 159,999 rows -> data\XRPUSDT\M5\ohlcv.csv
  H1: 39,999 rows -> data\XRPUSDT\H1\ohlcv.csv
  D1: 1,829 rows -> data\XRPUSDT\D1\ohlcv.csv
Fetching ADAUSDT...
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
    attempt 2/5 failed (('Connection aborted.', ConnectionResetEr

2026-05-17 09:00:09 - pybit._http_manager - ERROR - Too many visits. Exceeded the API Rate Limit. (ErrCode: 10006). Hit the API rate limit on https://api.bybit.com/v5/market/kline?category=linear&end=1765195799999&interval=5&limit=1000&symbol=ADAUSDT. Sleeping then trying again.
2026-05-17 09:00:09 - pybit._http_manager - ERROR - API rate limit will reset at 09:00:11.050. Sleeping for 2000 ms. Retrying...


    attempt 1/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Read timed out.), retrying in 2s...
  M5: 159,999 rows -> data\ADAUSDT\M5\ohlcv.csv
  H1: 39,999 rows -> data\ADAUSDT\H1\ohlcv.csv
  D1: 1,886 rows -> data\ADAUSDT\D1\ohlcv.csv
Fetching SOLUSDT...
    attempt 1/5 failed (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))), retrying in 2s...
    attempt 2/5 failed (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))), retrying in 4s...
    attempt 3/5 failed (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))), retrying in 8s...
    attempt 4/5 failed (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))), retrying in 16s...
 

2026-05-17 09:30:11 - pybit._http_manager - ERROR - Too many visits. Exceeded the API Rate Limit. (ErrCode: 10006). Hit the API rate limit on https://api.bybit.com/v5/market/kline?category=linear&end=1778877659999&interval=1&limit=1000&symbol=AVAXUSDT. Sleeping then trying again.
2026-05-17 09:30:11 - pybit._http_manager - ERROR - API rate limit will reset at 09:30:13.073. Sleeping for 2000 ms. Retrying...
2026-05-17 09:30:15 - pybit._http_manager - ERROR - Too many visits. Exceeded the API Rate Limit. (ErrCode: 10006). Hit the API rate limit on https://api.bybit.com/v5/market/kline?category=linear&end=1778757659999&interval=1&limit=1000&symbol=AVAXUSDT. Sleeping then trying again.
2026-05-17 09:30:15 - pybit._http_manager - ERROR - API rate limit will reset at 09:30:17.333. Sleeping for 2000 ms. Retrying...
2026-05-17 09:30:20 - pybit._http_manager - ERROR - Too many visits. Exceeded the API Rate Limit. (ErrCode: 10006). Hit the API rate limit on https://api.bybit.com/v5/market/kline?

    attempt 1/5 failed (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))), retrying in 2s...
    attempt 2/5 failed (HTTPSConnectionPool(host='api.bybit.com', port=443): Max retries exceeded with url: /v5/market/kline?category=linear&end=1766937659999&interval=1&limit=1000&symbol=AVAXUSDT (Caused by ConnectTimeoutError(<HTTPSConnection(host='api.bybit.com', port=443) at 0x1dc8a72e990>, 'Connection to api.bybit.com timed out. (connect timeout=10)'))), retrying in 4s...
    attempt 3/5 failed (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))), retrying in 8s...
    attempt 4/5 failed (('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))), retrying in 16s...
  M1: FAILED — Failed after 5 attempts for AVAXUSDT 1: ('Connection aborted.

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))